# Audi Used Car Price Prediction

**Objective:** Predict the price of a used Audi car using its specifications.

**Dataset:** `audi.csv` — 10,668 rows, 9 columns (`model`, `year`, `price`, `transmission`, `mileage`, `fuelType`, `tax`, `mpg`, `engineSize`)

**Approach (as taught in class):**
1. Load & explore the data
2. Split into features (X) and target (Y)
3. Encode categorical columns — Label Encoding (`model`, `fuelType`) + One-Hot Encoding (`transmission`)
4. Feature scaling with StandardScaler
5. Train-test split
6. Train four models — Random Forest, Linear Regression, Extra Trees, CatBoost
7. Evaluate each model on the test set **and** on the full dataset
8. Hyperparameter tuning of Random Forest using RandomizedSearchCV
9. Final comparison table of all models

## 1. Import Libraries

In [1]:
import pandas as pd
import numpy as np

## 2. Load and Explore the Dataset

In [2]:
df = pd.read_csv('audi.csv')
print(len(df))

10668


In [3]:
display(df.shape)

(10668, 9)

In [4]:
display(df.dtypes)

model            object
year              int64
price             int64
transmission     object
mileage           int64
fuelType         object
tax               int64
mpg             float64
engineSize      float64
dtype: object

In [5]:
display(df.isna().sum())

model           0
year            0
price           0
transmission    0
mileage         0
fuelType        0
tax             0
mpg             0
engineSize      0
dtype: int64

In [6]:
print(df.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10668 entries, 0 to 10667
Data columns (total 9 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   model         10668 non-null  object 
 1   year          10668 non-null  int64  
 2   price         10668 non-null  int64  
 3   transmission  10668 non-null  object 
 4   mileage       10668 non-null  int64  
 5   fuelType      10668 non-null  object 
 6   tax           10668 non-null  int64  
 7   mpg           10668 non-null  float64
 8   engineSize    10668 non-null  float64
dtypes: float64(2), int64(4), object(3)
memory usage: 750.2+ KB
None


In [7]:
# Check for duplicate rows
print(df[df.duplicated()])

     model  year  price transmission  mileage fuelType  tax   mpg  engineSize
273     Q3  2019  34485    Automatic       10   Diesel  145  47.1         2.0
764     Q2  2019  22495       Manual     1000   Diesel  145  49.6         1.6
784     Q3  2015  13995       Manual    35446   Diesel  145  54.3         2.0
967     Q5  2019  31998    Semi-Auto      100   Petrol  145  33.2         2.0
990     Q2  2019  22495       Manual     1000   Diesel  145  49.6         1.6
...    ...   ...    ...          ...      ...      ...  ...   ...         ...
9508    A4  2019  26990    Automatic     2250   Diesel  145  50.4         2.0
9521    Q3  2019  26990       Manual       10   Petrol  145  40.9         1.5
9529    Q5  2019  44990    Automatic       10   Diesel  145  36.2         2.0
9550    Q3  2019  29995       Manual       10   Petrol  145  39.8         1.5
9597    Q3  2019  28490       Manual       10   Diesel  145  42.8         2.0

[103 rows x 9 columns]


In [8]:
display(df.describe())

,year,price,mileage,tax,mpg,engineSize
count,10668.000000,10668.000000,10668.000000,10668.000000,10668.000000,10668.000000
mean,2017.100675,22896.685039,24827.244001,126.011436,50.770022,1.930709
std,2.167494,11714.841888,23505.257205,67.170294,12.949782,0.602957
min,1997.000000,1490.000000,1.000000,0.000000,18.900000,0.000000
25%,2016.000000,15130.750000,5968.750000,125.000000,40.900000,1.500000
50%,2017.000000,20200.000000,19000.000000,145.000000,49.600000,2.000000
75%,2019.000000,27990.000000,36464.500000,145.000000,58.900000,2.000000
max,2020.000000,145000.000000,323000.000000,580.000000,188.300000,6.300000


In [9]:
df

,model,year,price,transmission,mileage,fuelType,tax,mpg,engineSize
0,A1,2017,12500,Manual,15735,Petrol,150,55.4,1.4
1,A6,2016,16500,Automatic,36203,Diesel,20,64.2,2.0
2,A1,2016,11000,Manual,29946,Petrol,30,55.4,1.4
3,A4,2017,16800,Automatic,25952,Diesel,145,67.3,2.0
4,A3,2019,17300,Manual,1998,Petrol,145,49.6,1.0
...,...,...,...,...,...,...,...,...,...
10663,A3,2020,16999,Manual,4018,Petrol,145,49.6,1.0
10664,A3,2020,16999,Manual,1978,Petrol,150,49.6,1.0
10665,A3,2020,17199,Manual,609,Petrol,150,49.6,1.0
10666,Q3,2017,19499,Automatic,8646,Petrol,150,47.9,1.4


## 2b. Automated EDA Report (Graphs)

Instead of writing each chart by hand, we use `ydata_profiling` to auto-generate a full visual report —
distributions, correlations, missing values, and duplicate rows — in one step.

In [10]:
import ydata_profiling as pf
profile = pf.ProfileReport(df, title="Audi Used Car - EDA Report")
profile

/tmp/ipykernel_551/1677494698.py:1: DeprecationWarning: 
    `import ydata_profiling` is deprecated and will not receive more updates. 
    Please install fg-data-profiling via `pip install fg-data-profiling` and use `import data_profiling` instead.
    
  import ydata_profiling as pf


Summarize dataset:   0%|          | 0/5 [00:00<?, ?it/s]

  0%|          | 0/9 [00:00<?, ?it/s]

 67%|██████▋   | 6/9 [00:00<00:00, 48.81it/s]

100%|██████████| 9/9 [00:00<00:00, 58.77it/s]

Generate report structure:   0%|          | 0/1 [00:00<?, ?it/s]

Render HTML:   0%|          | 0/1 [00:00<?, ?it/s]

## 3. Split into Features (X) and Target (Y)

In [11]:
# X = all columns except price (index 2)
X = df.iloc[:, [0, 1, 3, 4, 5, 6, 7, 8]].values
display(X.shape)
display(X)

(10668, 8)

array([[' A1', 2017, 'Manual', ..., 150, 55.4, 1.4],
       [' A6', 2016, 'Automatic', ..., 20, 64.2, 2.0],
       [' A1', 2016, 'Manual', ..., 30, 55.4, 1.4],
       ...,
       [' A3', 2020, 'Manual', ..., 150, 49.6, 1.0],
       [' Q3', 2017, 'Automatic', ..., 150, 47.9, 1.4],
       [' Q3', 2016, 'Manual', ..., 150, 47.9, 1.4]],
      shape=(10668, 8), dtype=object)

In [12]:
# Y = price column
Y = df.iloc[:, 2].values
display(Y.shape)
display(Y)

(10668,)

array([12500, 16500, 11000, ..., 17199, 19499, 15999], shape=(10668,))

In [13]:
display(pd.DataFrame(X).head(5))

,0,1,2,3,4,5,6,7
0,A1,2017,Manual,15735,Petrol,150,55.4,1.4
1,A6,2016,Automatic,36203,Diesel,20,64.2,2.0
2,A1,2016,Manual,29946,Petrol,30,55.4,1.4
3,A4,2017,Automatic,25952,Diesel,145,67.3,2.0
4,A3,2019,Manual,1998,Petrol,145,49.6,1.0


## 4. Encoding Categorical Variables

In [14]:
# Label Encoding for 'model' (column 0) and 'fuelType' (column 4)
from sklearn.preprocessing import LabelEncoder

le1 = LabelEncoder()
X[:, 0] = le1.fit_transform(X[:, 0])

le2 = LabelEncoder()
X[:, 4] = le2.fit_transform(X[:, 4])

display(X)

array([[0, 2017, 'Manual', ..., 150, 55.4, 1.4],
       [5, 2016, 'Automatic', ..., 20, 64.2, 2.0],
       [0, 2016, 'Manual', ..., 30, 55.4, 1.4],
       ...,
       [2, 2020, 'Manual', ..., 150, 49.6, 1.0],
       [9, 2017, 'Automatic', ..., 150, 47.9, 1.4],
       [9, 2016, 'Manual', ..., 150, 47.9, 1.4]],
      shape=(10668, 8), dtype=object)

In [15]:
# One-Hot Encoding for 'transmission' (column 2)
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer

ct = ColumnTransformer(transformers=[('encoder', OneHotEncoder(), [2])], remainder='passthrough')
X = ct.fit_transform(X)
display(X.shape)
display(pd.DataFrame(X))

(10668, 10)

,0,1,2,3,4,5,6,7,8,9
0,0.0,1.0,0.0,0,2017,15735,2,150,55.4,1.4
1,1.0,0.0,0.0,5,2016,36203,0,20,64.2,2.0
2,0.0,1.0,0.0,0,2016,29946,2,30,55.4,1.4
3,1.0,0.0,0.0,3,2017,25952,0,145,67.3,2.0
4,0.0,1.0,0.0,2,2019,1998,2,145,49.6,1.0
...,...,...,...,...,...,...,...,...,...,...
10663,0.0,1.0,0.0,2,2020,4018,2,145,49.6,1.0
10664,0.0,1.0,0.0,2,2020,1978,2,150,49.6,1.0
10665,0.0,1.0,0.0,2,2020,609,2,150,49.6,1.0
10666,1.0,0.0,0.0,9,2017,8646,2,150,47.9,1.4


## 5. Feature Scaling

In [16]:
from sklearn.preprocessing import StandardScaler

sc = StandardScaler()
X = sc.fit_transform(X)
display(pd.DataFrame(X))

,0,1,2,3,4,5,6,7,8,9
0,-0.583268,1.200728,-0.712333,-1.123544,-0.046450,-0.386836,1.050783,0.357147,0.357550,-0.880218
1,1.714479,-0.832828,-0.712333,-0.160831,-0.507834,0.483989,-0.954181,-1.578323,1.037130,0.114925
2,-0.583268,1.200728,-0.712333,-1.123544,-0.507834,0.217781,1.050783,-1.429440,0.357550,-0.880218
3,1.714479,-0.832828,-0.712333,-0.545916,-0.046450,0.047853,-0.954181,0.282706,1.276528,0.114925
4,-0.583268,1.200728,-0.712333,-0.738459,0.876318,-0.971285,1.050783,0.282706,-0.090355,-1.543647
...,...,...,...,...,...,...,...,...,...,...
10663,-0.583268,1.200728,-0.712333,-0.738459,1.337702,-0.885343,1.050783,0.282706,-0.090355,-1.543647
10664,-0.583268,1.200728,-0.712333,-0.738459,1.337702,-0.972136,1.050783,0.357147,-0.090355,-1.543647
10665,-0.583268,1.200728,-0.712333,-0.738459,1.337702,-1.030381,1.050783,0.357147,-0.090355,-1.543647
10666,1.714479,-0.832828,-0.712333,0.609339,-0.046450,-0.688442,1.050783,0.357147,-0.221637,-0.880218


## 6. Train-Test Split

In [17]:
from sklearn.model_selection import train_test_split

X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.2, random_state=0)
print(X.shape, Y.shape)
print(X_train.shape, Y_train.shape)
print(X_test.shape, Y_test.shape)

(10668, 10) (10668,)
(8534, 10) (8534,)
(2134, 10) (2134,)


## 7. Model 1 — Random Forest Regressor

In [18]:
from sklearn.ensemble import RandomForestRegressor

regression = RandomForestRegressor(random_state=0)
regression.fit(X_train, Y_train)
display(regression)

,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",100
,"criterion criterion: {""squared_error"", ""absolute_error"", ""friedman_mse"", ""poisson""}, default=""squared_error""The function to measure the quality of a split. Supported criteriaare ""squared_error"" for the mean squared error, which is equal tovariance reduction as feature selection criterion and minimizes the L2loss using the mean of each terminal node, ""friedman_mse"", which usesmean squared error with Friedman's improvement score for potentialsplits, ""absolute_error"" for the mean absolute error, which minimizesthe L1 loss using the median of each terminal node, and ""poisson"" whichuses reduction in Poisson deviance to find splits.Training using ""absolute_error"" is significantly slowerthan when using ""squared_error""... versionadded:: 0.18 Mean Absolute Error (MAE) criterion... versionadded:: 1.0 Poisson criterion.",'squared_error'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=1.0The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None or 1.0, then `max_features=n_features`... note:: The default of 1.0 is equivalent to bagged trees and more randomness can be achieved by setting smaller values, e.g. 0.3... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to 1.0.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",1.0
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsample

In [19]:
y_pred = regression.predict(X_test)
display(y_pred)

array([14337.15, 23450.35, 27330.07, ..., 46275.18, 31359.  ,  9929.62],
      shape=(2134,))

In [20]:
print(pd.concat([pd.DataFrame(Y_test.reshape(len(Y_test), 1), columns=['Car Price']),
                  pd.DataFrame(y_pred.reshape(len(y_pred), 1), columns=['Price Prediction'])], axis=1))

      Car Price  Price Prediction
0         14998          14337.15
1         21950          23450.35
2         28990          27330.07
3         25489          27200.98
4         30950          32250.05
...         ...               ...
2129      23700          39147.77
2130      18000          16679.95
2131      45995          46275.18
2132      30500          31359.00
2133       8400           9929.62

[2134 rows x 2 columns]


In [21]:
from sklearn.metrics import r2_score, mean_absolute_error

rf_test_r2 = r2_score(Y_test, y_pred)
rf_test_mae = mean_absolute_error(Y_test, y_pred)
print('R2 Score ', rf_test_r2)
print('Mean Absolute Error', rf_test_mae)

R2 Score  0.9536134841307546
Mean Absolute Error 1538.730980670462


**Random Forest — predictions on the full dataset**

In [22]:
f_pred = regression.predict(X)
display(f_pred)

array([12864.85, 16457.12, 11755.03, ..., 17642.77, 20217.97, 18360.66],
      shape=(10668,))

In [23]:
display(pd.concat([df, pd.DataFrame(f_pred.reshape(len(f_pred), 1), columns=['Price Prediction'])], axis=1))

,model,year,price,transmission,mileage,fuelType,tax,mpg,engineSize,Price Prediction
0,A1,2017,12500,Manual,15735,Petrol,150,55.4,1.4,12864.85
1,A6,2016,16500,Automatic,36203,Diesel,20,64.2,2.0,16457.12
2,A1,2016,11000,Manual,29946,Petrol,30,55.4,1.4,11755.03
3,A4,2017,16800,Automatic,25952,Diesel,145,67.3,2.0,17664.67
4,A3,2019,17300,Manual,1998,Petrol,145,49.6,1.0,17191.84
...,...,...,...,...,...,...,...,...,...,...
10663,A3,2020,16999,Manual,4018,Petrol,145,49.6,1.0,17052.35
10664,A3,2020,16999,Manual,1978,Petrol,150,49.6,1.0,16985.51
10665,A3,2020,17199,Manual,609,Petrol,150,49.6,1.0,17642.77
10666,Q3,2017,19499,Automatic,8646,Petrol,150,47.9,1.4,20217.97


In [24]:
rf_full_r2 = r2_score(Y, f_pred)
rf_full_mae = mean_absolute_error(Y, f_pred)
print('R2 Score ', rf_full_r2)
print('Mean Absolute Error', rf_full_mae)

R2 Score  0.9854936957854944
Mean Absolute Error 784.8616110361776


## 8. Model 2 — Linear Regression

In [25]:
from sklearn.linear_model import LinearRegression

reg = LinearRegression()
reg.fit(X_train, Y_train)
display(reg)

,"fit_intercept fit_intercept: bool, default=TrueWhether to calculate the intercept for this model. If setto False, no intercept will be used in calculations(i.e. data is expected to be centered).",True
,"copy_X copy_X: bool, default=TrueIf True, X will be copied; else, it may be overwritten.",True
,"tol tol: float, default=1e-6The precision of the solution (`coef_`) is determined by `tol` whichspecifies a different convergence criterion for the `lsqr` solver.`tol` is set as `atol` and `btol` of :func:`scipy.sparse.linalg.lsqr` whenfitting on sparse training data. This parameter has no effect when fittingon dense data... versionadded:: 1.7",1e-06
,"n_jobs n_jobs: int, default=NoneThe number of jobs to use for the computation. This will only providespeedup in case of sufficiently large problems, that is if firstly`n_targets > 1` and secondly `X` is sparse or if `positive` is setto `True`. ``None`` means 1 unless in a:obj:`joblib.parallel_backend` context. ``-1`` means using allprocessors. See :term:`Glossary ` for more details.",None
,"positive positive: bool, default=FalseWhen set to ``True``, forces the coefficients to be positive. Thisoption is only supported for dense arrays.For a comparison between a linear regression model with positive constraintson the regression coefficients and a linear regression without such constraints,see :ref:`sphx_glr_auto_examples_linear_model_plot_nnls.py`... versionadded:: 0.24",False


In [26]:
y_pred = reg.predict(X_test)
display(y_pred)

array([13052.36575395, 29340.03009442, 31908.12716369, ...,
       42649.94612604, 31554.51438191,  7285.29079975], shape=(2134,))

In [27]:
print(pd.concat([pd.DataFrame(Y_test.reshape(len(Y_test), 1), columns=['Car Price']),
                  pd.DataFrame(y_pred.reshape(len(y_pred), 1), columns=['Price Prediction'])], axis=1))

      Car Price  Price Prediction
0         14998      13052.365754
1         21950      29340.030094
2         28990      31908.127164
3         25489      26834.582322
4         30950      31477.717507
...         ...               ...
2129      23700      41546.889475
2130      18000      20705.470993
2131      45995      42649.946126
2132      30500      31554.514382
2133       8400       7285.290800

[2134 rows x 2 columns]


In [28]:
lr_test_r2 = r2_score(Y_test, y_pred)
lr_test_mae = mean_absolute_error(Y_test, y_pred)
print('R2 Score ', lr_test_r2)
print('Mean Absolute Error', lr_test_mae)

R2 Score  0.7916214142758096
Mean Absolute Error 3381.673339589059


**Linear Regression — predictions on the full dataset**

In [29]:
f_pred = reg.predict(X)
display(pd.concat([df, pd.DataFrame(f_pred.reshape(len(f_pred), 1), columns=['Price Prediction'])], axis=1))

,model,year,price,transmission,mileage,fuelType,tax,mpg,engineSize,Price Prediction
0,A1,2017,12500,Manual,15735,Petrol,150,55.4,1.4,14619.723522
1,A6,2016,16500,Automatic,36203,Diesel,20,64.2,2.0,20635.866607
2,A1,2016,11000,Manual,29946,Petrol,30,55.4,1.4,13800.630701
3,A4,2017,16800,Automatic,25952,Diesel,145,67.3,2.0,19914.076834
4,A3,2019,17300,Manual,1998,Petrol,145,49.6,1.0,17339.679474
...,...,...,...,...,...,...,...,...,...,...
10663,A3,2020,16999,Manual,4018,Petrol,145,49.6,1.0,19179.618582
10664,A3,2020,16999,Manual,1978,Petrol,150,49.6,1.0,19266.676532
10665,A3,2020,17199,Manual,609,Petrol,150,49.6,1.0,19396.772612
10666,Q3,2017,19499,Automatic,8646,Petrol,150,47.9,1.4,20998.775674


In [30]:
lr_full_r2 = r2_score(Y, f_pred)
lr_full_mae = mean_absolute_error(Y, f_pred)
print('R2 Score ', lr_full_r2)
print('Mean Absolute Error', lr_full_mae)

R2 Score  0.79071881497407
Mean Absolute Error 3344.046507014762


## 9. Model 3 — Extra Trees Regressor

In [31]:
from sklearn.ensemble import ExtraTreesRegressor

ET_Model = ExtraTreesRegressor(n_estimators=120, random_state=0)
ET_Model.fit(X_train, Y_train)
y_predict = ET_Model.predict(X_test)

et_test_r2 = r2_score(Y_test, y_predict)
et_test_mae = mean_absolute_error(Y_test, y_predict)
print('R2 Score ', et_test_r2)
print('Mean Absolute Error', et_test_mae)

R2 Score  0.957066105930189
Mean Absolute Error 1539.1081453712384


**Extra Trees — predictions on the full dataset**

In [32]:
full_et_pred = ET_Model.predict(X)

et_full_r2 = r2_score(Y, full_et_pred)
et_full_mae = mean_absolute_error(Y, full_et_pred)
print('R2 Score ', et_full_r2)
print('Mean Absolute Error', et_full_mae)

R2 Score  0.9904972371623373
Mean Absolute Error 350.65945340165814


## 10. Model 4 — CatBoost Regressor

In [33]:
from catboost import CatBoostRegressor

cat = CatBoostRegressor(verbose=False)
cat.fit(X_train, Y_train)
display(cat)

CatBoostRegressor(loss_function='RMSE', verbose=False)

In [34]:
cat_pred = cat.predict(X_test)
display(cat_pred)

array([13386.63817795, 24056.86842882, 28082.99514751, ...,
       45959.31234642, 31714.44751009,  9481.45994163], shape=(2134,))

In [35]:
cat_test_r2 = r2_score(Y_test, cat_pred)
cat_test_mae = mean_absolute_error(Y_test, cat_pred)
print('Mean Absolute Error', cat_test_mae)
print('R2 Score ', cat_test_r2)

Mean Absolute Error 1453.5230700749125
R2 Score  0.9641612028134969


**CatBoost — predictions on the full dataset**

In [36]:
catf_pred = cat.predict(X)
display(catf_pred)

array([13989.8267214 , 16578.03846793, 12489.54800057, ...,
       18716.55396652, 20908.57305802, 17896.99245815], shape=(10668,))

In [37]:
result = pd.concat([df, pd.DataFrame(catf_pred, columns=['Price Prediction'])], axis=1)
display(result)

,model,year,price,transmission,mileage,fuelType,tax,mpg,engineSize,Price Prediction
0,A1,2017,12500,Manual,15735,Petrol,150,55.4,1.4,13989.826721
1,A6,2016,16500,Automatic,36203,Diesel,20,64.2,2.0,16578.038468
2,A1,2016,11000,Manual,29946,Petrol,30,55.4,1.4,12489.548001
3,A4,2017,16800,Automatic,25952,Diesel,145,67.3,2.0,18793.884463
4,A3,2019,17300,Manual,1998,Petrol,145,49.6,1.0,18493.205301
...,...,...,...,...,...,...,...,...,...,...
10663,A3,2020,16999,Manual,4018,Petrol,145,49.6,1.0,19428.713314
10664,A3,2020,16999,Manual,1978,Petrol,150,49.6,1.0,18877.541748
10665,A3,2020,17199,Manual,609,Petrol,150,49.6,1.0,18716.553967
10666,Q3,2017,19499,Automatic,8646,Petrol,150,47.9,1.4,20908.573058


In [38]:
cat_full_r2 = r2_score(Y, catf_pred)
cat_full_mae = mean_absolute_error(Y, catf_pred)
print('R2 Score ', cat_full_r2)
print('Mean Absolute Error', cat_full_mae)

R2 Score  0.9745608180217411
Mean Absolute Error 1300.9813014378083


## 11. Hyperparameter Tuning — Random Forest (RandomizedSearchCV)

> **Note:** The full search grid used in class (`n_iter=200`, trees up to 1500, `cv=5`) can take a very long time to run.
> The method below is identical — a `RandomizedSearchCV` tuning a `RandomForestRegressor` — but with a lighter
> search space so it completes in a reasonable time.

In [39]:
from sklearn.model_selection import RandomizedSearchCV

n_estimators = [int(x) for x in np.linspace(start=80, stop=400, num=6)]
max_features = ['sqrt', 'log2']
max_depth = [int(x) for x in np.linspace(6, 45, num=4)]
min_samples_split = [2, 5, 10]
min_samples_leaf = [1, 2, 4]

rand_grid = {'n_estimators': n_estimators,
             'max_features': max_features,
             'max_depth': max_depth,
             'min_samples_split': min_samples_split,
             'min_samples_leaf': min_samples_leaf}

rf = RandomForestRegressor(random_state=0)
rCV = RandomizedSearchCV(estimator=rf, param_distributions=rand_grid, scoring='neg_mean_squared_error',
                          n_iter=15, cv=3, random_state=42, n_jobs=-1)
rCV.fit(X_train, Y_train)
print(rCV.best_params_)

{'n_estimators': 336, 'min_samples_split': 5, 'min_samples_leaf': 1, 'max_features': 'sqrt', 'max_depth': 45}


In [40]:
rf_pred = rCV.predict(X_test)
display(rf_pred)

array([14063.75609249, 23623.46185677, 28713.41850789, ...,
       48607.72648201, 31089.37638535, 10210.14689862], shape=(2134,))

In [41]:
from sklearn.metrics import mean_squared_error

rf_tuned_test_r2 = r2_score(Y_test, rf_pred)
rf_tuned_test_mae = mean_absolute_error(Y_test, rf_pred)
print('MAE', rf_tuned_test_mae)
print('MSE', mean_squared_error(Y_test, rf_pred))
print('R2 Score ', rf_tuned_test_r2)

MAE 1493.420339192943
MSE 5558546.577861906
R2 Score  0.9595494987202554


**Tuned Random Forest — predictions on the full dataset**

In [42]:
full_rf_pred = rCV.predict(X)
display(full_rf_pred)

array([13127.20383909, 16185.62121931, 11777.00865158, ...,
       17772.80629864, 20435.8873018 , 18510.36120087], shape=(10668,))

In [43]:
rf_tuned_full_r2 = r2_score(Y, full_rf_pred)
rf_tuned_full_mae = mean_absolute_error(Y, full_rf_pred)
print('MAE', rf_tuned_full_mae)
print('MSE', mean_squared_error(Y, full_rf_pred))
print('R2 Score ', rf_tuned_full_r2)

MAE 1019.8521037350407
MSE 2602745.105106175
R2 Score  0.981032963172884


## 12. Final Model Comparison

In [44]:
summary = pd.DataFrame({
    'Model': ['Random Forest', 'Linear Regression', 'Extra Trees', 'CatBoost', 'Random Forest (Tuned)'],
    'Test ACC (%)': [round(rf_test_r2*100), round(lr_test_r2*100), round(et_test_r2*100),
                      round(cat_test_r2*100), round(rf_tuned_test_r2*100)],
    'Test MAE': [round(rf_test_mae), round(lr_test_mae), round(et_test_mae),
                 round(cat_test_mae), round(rf_tuned_test_mae)],
    'Full-Data ACC (%)': [round(rf_full_r2*100), round(lr_full_r2*100), round(et_full_r2*100),
                           round(cat_full_r2*100), round(rf_tuned_full_r2*100)],
    'Full-Data MAE': [round(rf_full_mae), round(lr_full_mae), round(et_full_mae),
                       round(cat_full_mae), round(rf_tuned_full_mae)]
})
display(summary)

,Model,Test ACC (%),Test MAE,Full-Data ACC (%),Full-Data MAE
0,Random Forest,95,1539,99,785
1,Linear Regression,79,3382,79,3344
2,Extra Trees,96,1539,99,351
3,CatBoost,96,1454,97,1301
4,Random Forest (Tuned),96,1493,98,1020


## 13. Conclusion

- Four regression models were trained and compared — **Random Forest**, **Linear Regression**, **Extra Trees**, and **CatBoost** — each evaluated on both the held-out test set and the full dataset.
- **Linear Regression** performs noticeably worse (R² ≈ 0.79) than the tree-based models, since car price does not vary linearly with mileage, age, and engine size.
- **Random Forest, Extra Trees, and CatBoost** all perform strongly (R² between 0.95–0.99), with Extra Trees achieving the lowest error on the full dataset.
- Hyperparameter tuning of Random Forest via `RandomizedSearchCV` was used to search for a better parameter combination; the tuned model's performance is shown in the final row of the summary table above.
- The final comparison table summarizes test-set and full-data accuracy/error for all five model variants, matching the evaluation format used in class.